# 03 — RAG Üretimi ve Yanıt Değerlendirme (İP-4, İP-6)

Kazanılan erişim sonuçlarını sabit LLM'e bağlar: her sistem için Top-k kaynak + yanıt üretir, yanıtları kaydeder.

**Değişmez kural:** Tüm sistemlerde aynı LLM, prompt (`prompts/rag_tr.txt`), Top-k ve üretim ayarları. Fark yalnızca erişim katmanından gelir (RQ3).

In [ ]:
import os, sys
REPO_URL = 'https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri.git'
BRANCH = 'RAG-dev'
REPO_DIR = '/content/RAG-proje'
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull origin {BRANCH}
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

from src.config import load_config
from src.data import jsonl_oku, jsonl_yaz, korpus_yukle, yonetmelik_parcala
from src.embeddings import DenseEmbedder
from src.retrieval import Bm25Retriever, FaissRetriever
from src.generation import RagGenerator

cfg = load_config()
chunklar = yonetmelik_parcala(
    korpus_yukle(cfg['veri']['korpus']),
    boyut=cfg['parcalama']['boyut_karakter'],
    ortusme=cfg['parcalama']['ortusme_karakter'],
)
chunk_metin = {c.chunk_id: c.metin for c in chunklar}
sorular = [s for s in jsonl_oku(cfg['veri']['sorular']) if s.get('bolme', 'test') == 'test']
# Her soru tipinden (olgusal, paraphrase, bicimbilimsel, cok-maddeli) dengeli 5'er soru = 20 test sorusu
ornek_sorular = []
for tip in ['olgusal', 'paraphrase', 'bicimbilimsel', 'cok-maddeli']:
    ornek_sorular.extend([s for s in sorular if s['tip'] == tip][:5])
print(f'Toplam test sorusu: {len(sorular)} | RAG üretim örneklemi (dengeli): {len(ornek_sorular)}')

In [ ]:
# Üreticiyi yükle (4-bit; ilk seferde model indirilir, ~5 GB)
uretici = RagGenerator(
    model_adi=cfg['uretim']['llm'],
    dort_bit=(cfg['uretim']['yukleme'] == '4bit'),
    maks_yeni_token=cfg['uretim']['maks_yeni_token'],
)
print('LLM hazır:', cfg['uretim']['llm'])

In [ ]:
# RQ3 için 4 temsilci erişim sistemi (bm25, minilm, tr-model, e5-base) + sabit LLM ile yanıt üretimi
secili_modeller = ['bm25', 'minilm', 'tr-model', 'e5-base']
sistemler = {}
embedder_map = {}
for m in cfg['modeller']:
    ad = m['ad']
    if ad not in secili_modeller:
        continue
    if m['tur'] == 'sozcuksel':
        sistemler[ad] = Bm25Retriever(chunklar)
    else:
        emb = DenseEmbedder(
            m['hf'],
            sorgu_oneki=m.get('sorgu_oneki', ''),
            belge_oneki=m.get('belge_oneki', ''),
            sorgu_yonergesi=m.get('sorgu_yonergesi', ''),
        )
        embedder_map[ad] = emb
        sistemler[ad] = FaissRetriever(emb, chunklar)

top_k = cfg['erisim']['ana_top_k']
kayitlar = []
for ad, sistem in sistemler.items():
    print(f'=== Üretim başlıyor: {ad} ({len(ornek_sorular)} soru) ===')
    for soru in ornek_sorular:
        bulunan = sistem.retrieve(soru['soru'], k=top_k)
        kaynak_idler = [cid for cid, _ in bulunan]
        kaynaklar = [chunk_metin[cid] for cid in kaynak_idler]
        yanit = uretici.uret(soru['soru'], kaynaklar)
        kayitlar.append({
            'soru_id': soru['soru_id'],
            'tip': soru['tip'],
            'sistem': ad,
            'soru': soru['soru'],
            'kanit_chunk_id': soru['kanit_chunk_id'],
            'kaynaklar': kaynak_idler,
            'hit@5': 1.0 if soru['kanit_chunk_id'] in kaynak_idler else 0.0,
            'yanit': yanit,
            'referans': soru.get('referans_yanit', ''),
        })
    print(f'{ad} örnek yanıt ({ornek_sorular[0]["soru_id"]}):', kayitlar[-len(ornek_sorular)]['yanit'][:120], '...')

jsonl_yaz(kayitlar, 'results/yanitlar-rq3.jsonl')
print(f'Toplam {len(kayitlar)} yanıt kaydedildi: results/yanitlar-rq3.jsonl')

## Yanıt değerlendirme (faithfulness)

- Üretilen her yanıt için kaynaklar kaydedildiği için faithfulness, yanıtın iddialarının kaynaklarda desteklenip desteklenmediğine bakarak ölçülür.
- Pratik yol: `ragas` ile LLM-as-judge (faithfulness + answer correctness). **Judge modeli, üretim LLM'inden farklı ve sabit sürümlü olmalı**; hangi modelin judge olduğu rapora yazılır.
- RQ3 analizi: soru bazında retrieval skoru (notebook 02) ile yanıt doğruluğunu eşleştirip korelasyon bakılır; doğrusal olmayan ilişki beklenen bulgudur (H3).

In [ ]:
# RQ3 Değerlendirmesi: Faithfulness (Kaynağa Bağlılık), Answer Correctness ve Atıf Doğruluğu
import re
import numpy as np
import pandas as pd

_TOK = re.compile(r'\w+', re.UNICODE)
_STOP = {'ve', 'ile', 'bir', 'bu', 'için', 'olarak', 'olan', 'göre', 'veya', 'da', 'de', 'ise', 'en', 'her', 'ne', 'kadar'}

def icerik_tokenlari(metin: str) -> set[str]:
    return {t for t in _TOK.findall(metin.lower()) if len(t) > 2 and t not in _STOP}

def kaynak_baglilik_skoru(yanit: str, kaynak_idler: list[str]) -> float:
    """Yanıttaki içerik terimlerinin getirilen kaynaklarda desteklenme oranı (faithfulness)."""
    if 'verilen kaynaklarda yer almıyor' in yanit.lower():
        return 1.0  # kaynakta yoksa doğru kaçınma (abstention) sadıktır
    y_tok = icerik_tokenlari(yanit)
    if not y_tok:
        return 0.0
    k_metin = ' '.join(chunk_metin[c] for c in kaynak_idler if c in chunk_metin)
    k_tok = icerik_tokenlari(k_metin)
    # Türkçe ek kök uyumu için ilk 5 karakter (gövde) eşleşmesi de kontrol edilir
    k_kokler = {t[:5] for t in k_tok if len(t) >= 5}
    desteklenen = sum(1 for t in y_tok if (t in k_tok) or (len(t) >= 5 and t[:5] in k_kokler))
    return desteklenen / len(y_tok)

def token_f1(yanit: str, referans: str) -> float:
    y_tok = icerik_tokenlari(yanit)
    r_tok = icerik_tokenlari(referans)
    if not y_tok or not r_tok:
        return 0.0
    y_kok = {t[:5] for t in y_tok if len(t) >= 5} | y_tok
    r_kok = {t[:5] for t in r_tok if len(t) >= 5} | r_tok
    ortak = len(y_kok & r_kok)
    if ortak == 0:
        return 0.0
    p = ortak / len(y_kok)
    r = ortak / len(r_kok)
    return 2 * p * r / (p + r)

# Anlamsal doğruluk için sabit değerlendirici (e5-base kosinüs benzerliği + sözcüksel F1 ortalaması)
judge_emb = embedder_map['e5-base']
yanit_vektorler = judge_emb.belgeleri_kodla([k['yanit'] for k in kayitlar])
ref_vektorler = judge_emb.belgeleri_kodla([k['referans'] for k in kayitlar])
anlamsal_skorlar = np.sum(yanit_vektorler * ref_vektorler, axis=1)

for k, sem_skor in zip(kayitlar, anlamsal_skorlar):
    k['faithfulness'] = round(float(kaynak_baglilik_skoru(k['yanit'], k['kaynaklar'])), 4)
    f1_skor = token_f1(k['yanit'], k['referans'])
    k['answer_correctness'] = round(float(0.6 * sem_skor + 0.4 * f1_skor), 4)
    hedef_madde = k['kanit_chunk_id'].split('-')[0] + '-' + k['kanit_chunk_id'].split('-')[1]
    k['dogru_atif'] = 1.0 if hedef_madde in k['yanit'].upper().replace(' ', '-') else 0.0

df_rq3 = pd.DataFrame(kayitlar)
rq3_ozet = df_rq3.groupby('sistem')[['hit@5', 'faithfulness', 'answer_correctness', 'dogru_atif']].mean().round(4)
rq3_ozet = rq3_ozet.sort_values('hit@5', ascending=False)
rq3_ozet.to_csv('results/rq3-rag-ozet.csv')
print('=== RQ3: Retrieval Başarısı vs Yanıt Kalitesi ve Faithfulness ===')
print(rq3_ozet)

print('\n=== Kanıt Bulunan (Hit@5=1) vs Bulunamayan (Hit@5=0) Sorularda Yanıt Kalitesi ===')
print(df_rq3.groupby('hit@5')[['faithfulness', 'answer_correctness', 'dogru_atif']].mean().round(4))